# gsea-lite 基因集富集分析工具演示报告

`gsea-lite` 是一个专为转录组数据分析设计的轻量级基因集富集分析 Python 工具库。本项目完整实现了两种生信核心富集分析算法：**过表达分析 (ORA)** 与 **简化基因集富集分析 (GSEA)**，并集成了多重假设检验校正（BH-FDR）与高质量的科研数据可视化功能。

## 1、小组成员

* **丁奕新**：数据输入与经典富集模块（`data_io.py`, `ora.py`）
* **夏奕林婷**：多重检验与高级富集模块（`stats_utils.py`, `gsea.py`）
* **于润湉**：图形绘制与软件工程集成（`plot.py`, `main.py`, `tests/`）

## 2、核心功能

1. **标准 GMT 文件解析**：手工实现高容错率的 GMT 格式文件解析器，支持空字符清洗与非规范行剔除。
2. **差异表达数据校验**：基于 `pandas` 读取真实 GEO/TCGA 差异表达结果，具备严格的核心列名（`Gene_Symbol`, `log2FoldChange`, `pvalue`）和空值过滤校验。
3. **经典过表达分析 (ORA)**：自定义全局背景基因库（Universe），利用 `scipy.stats` 超几何生存函数（Survival Function）精准计算每个通路的原始 $p$-value。
4. **Benjamini-Hochberg (BH) FDR 校正**：原生算法实现多重假设检验校正，通过严谨的递增保护逻辑确保 FDR 随着原 $p$-value 递增而保持绝对单调性。
5. **简化版 GSEA 富集分数计算**：根据表达变化幅度（$\log_2\text{FC}$）对全量基因进行降序排列，利用随机行走算法实时捕捉基因集在列表顶端或底端的最大偏离峰值（ES）。
6. **科研级图形可视化**：自动筛选 Top 10 显著富集通路，输出符合学术期刊发表标准的富集柱状图（横轴为 $-\log_{10}(\text{FDR})$）与多维映射气泡图（映射 Gene Ratio, Count 和 FDR）。

## 3、目录

```text
gsea-lite/
│
├── data/
│   ├── GSE42568.top.table.tsv          # 真实差异表达数据 
│   └── h.all.v2026.1.Hs.symbols.gmt    # 从MSigDB下载的KEGG/GO基因集文件
│
├── src/
│   ├── __init__.py            
│   ├── data_io.py             # 读取数据代码
│   ├── stats_utils.py         # FDR矫正代码（数据预处理）
│   ├── ora.py                 # 过表达分析与超几何检验代码
│   ├── gsea.py                # GSEA代码
│   └── plot.py                # 数据可视化代码
│
├── results/
│   ├── gsea_results.csv
│   ├── ora_results_fdr.csv
│   ├── ora_results.csv        # ORA 分析定量报表
│   ├── ora_dotplot.png        # 富集多维气泡图
│   └── ora_barplot.png        # 富集显著性柱状图
│   
├── tests/
│   └── test_core.py           # pytest 单元测试集（边界情况与数学逻辑校验）
│
├── main.py
│   
└── README.md
```

## 3、环境准备与模块导入

In [8]:
import os
import numpy as np
import pandas as pd
from scipy.stats import hypergeom
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import argparse

# 确保图片在 Notebook 中内嵌显示
%matplotlib inline

## data_io.py

In [9]:
"""
1. 解析GMT文件
2. 读取GEO的TSV文件中的差异表达数据
"""
def read_gmt(gmt_file_path):
    """
    解析GMT文件

    返回一个字典：
    key = 通路名称
    value = 该通路含有的基因列表
    """
    gene_sets = {} 
    
    # 对于 file 中的每一行 line 进行处理
    with open(gmt_file_path, 'r', encoding='utf-8') as file:
        for line in file: 
            line = line.strip() 

            if not line: 
                continue 
            
            # 按 \t分割，得到一个列表items 
            items = line.split('\t') 
             
            if len(items) < 3: 
                continue 
            
            # 提取列表中的通路名称、基因
            pathway_name = items[0]
            genes = items[2:] # items[1] 通常是描述信息，忽略 
            
            # 清洗基因列表并存入字典
            valid_genes = []            

            for g in genes:                   
                if g.strip():                
                    valid_genes.append(g.strip()) 

            gene_sets[pathway_name] = valid_genes 
            
    return gene_sets 

def read_geo(geo_file_path):
    """
    读取GEO的TSV文件中的关键数据
    返回:经过过滤的差异表达数据dataframe
    """
        
    # 读取数据 
    df = pd.read_csv(geo_file_path, sep='\t', encoding='utf-8') 
    
    # 验证是否包含核心列 
    required_columns = ['Gene.symbol', 'logFC', 'P.Value'] 
    for col in required_columns: 
        if col not in df.columns: 
            raise ValueError(f"error: 缺失列 {col}") 
                
    # 去除空行
    df = df.dropna(subset=required_columns) 
    
    return df

## stats_utils.py

In [10]:
"""
FDR-based p-value adjuster.
https://github.com/dceoy/fdra
"""

def bh_fdr_correction(pvalue):
    """
    Benjamini-Hochberg FDR correction

    Parameters
    ----------
    pvalue : array-like
        DEG结果表中的 pvalue 列

    Returns
    -------
    numpy.ndarray
        q-value (FDR)
    """

    pvalue = np.asarray(pvalue, dtype=float)

    if np.any((pvalue < 0) | (pvalue > 1)):
        raise ValueError("Invalid p-values")

    # 总检验数
    m = len(pvalue)

    # 按 pvalue 从小到大排序
    # np.argsort():返回排好序的元素在原数组中的索引（位置编号）
    sorted_index = np.argsort(pvalue) #返回排好序的元素在原数组pvalue中的位置编号
    sorted_pvalue = pvalue[sorted_index]

    # BH校正
    qvalue_sorted = sorted_pvalue * m / np.arange(1, m + 1)

    # 保证FDR单调递增
    # 一个基因的qvalue不能大于排在它后面的任何一个基因的qvalue。如果大于了，就得降到和后面一样低
    # np.minimum.accumulate()：沿给定轴计算元素的累积最小值，返回一个数组，其中每个位置包含到目前为止遇到的最小值。
    qvalue_sorted = np.minimum.accumulate(
        qvalue_sorted[::-1] #先反转，pvalue变成从大到小排
    )[::-1] #最后再反转回从小到大

    # 限制在[0,1]
    # np.clip(): 强制将所有算出来的qvalue限制在0到1的范围内。如果数组里有大于1的数，强制变成1。
    qvalue_sorted = np.clip(qvalue_sorted, 0, 1)

    # 恢复原顺序
    qvalue = np.empty(m)
    qvalue[sorted_index] = qvalue_sorted #qvalue_sorted是从小到大排好序的，sorted_index是原pvalue的索引

    return qvalue

## ora.py

In [11]:
def extract_significant_genes(df, p_thresh=0.05, lfc_thresh=1.0):
    """
    根据给定的p-value和|log2FC|阈值，从差异表达矩阵中筛选出显著差异基因DEGs
    """

    condition = (df['P.Value'] < p_thresh) & (df['logFC'].abs() > lfc_thresh)
    
    genes_filtered = df.loc[condition, 'Gene.symbol'] 
    
    sig_genes = set(genes_filtered.dropna().unique()) # 去除缺失值和重复基因，转换为集合,便于后续的交集运算
    
    return sig_genes

def do_ora(df, gene_sets, p_thresh=0.05, lfc_thresh=1.0):
    """
    ORA分析，计算每个通路的 p-value
    """
    
    # 1. 定义全局背景基因库
    universe_genes = set(df['Gene.symbol'].dropna().unique())
    M = len(universe_genes) 
    
    # 2. 提取显著差异基因集
    deg_genes = extract_significant_genes(df, p_thresh, lfc_thresh)
    deg_genes = deg_genes.intersection(universe_genes) # 确保差异基因一定在背景库中
    N = len(deg_genes) 

    print(f"--- 背景基因库说明 ---")
    print(f"提取到全局背景基因 (Universe) 数量: {M}")
    print(f"提取到显著差异基因 (DEGs) 数量: {N}")
    print(f"原理解释: 仅使用在当前测序平台/微阵列中实际检测到的 {M} 个基因作为超几何检验的背景，而不是整个人类基因组，能有效避免因技术偏倚导致的假阳性。")

    if N == 0:
        print("Warning: no significant genes found. ")
        return pd.DataFrame(columns=['pathway', 'P.Value', 'hits', 'pathway_size', 'hits_genes'])
        
    # 3. 遍历 GMT 字典中的每一个通路，进行超几何检验
    results = []
    
    for pathway_name, pathway_genes in gene_sets.items():
        pathway_set = set(pathway_genes)
        
        pathway_in_universe = pathway_set.intersection(universe_genes)
        n = len(pathway_in_universe)
        
        if n == 0:
            continue
            
        # 计算交集 k：既属于该通路，又是显著差异的基因
        hit_genes = deg_genes.intersection(pathway_in_universe)
        k = len(hit_genes)
        raw_p = hypergeom.sf(k - 1, M, n, N)
        
        # 【修改开始】
        results.append({
            'pathway': pathway_name,
            'P.Value': raw_p,
            'Count': k,                                  # 修改点 1：将键名 'hits' 改为 'Count'
            'pathway_size': n,
            'GeneRatio': k / N if N > 0 else 0,          # 修改点 2：根据公式 (命中基因数 / 差异基因总数) 计算 GeneRatio
            'hits_genes': ",".join(list(hit_genes))
        })
        # 【修改结束】
        
    ora_result_df = pd.DataFrame(results)
    
    # 按照原始 p-value 从小到大排列（最显著的排在最前）
    if not ora_result_df.empty:
        ora_result_df = ora_result_df.sort_values(by='P.Value').reset_index(drop=True)
        
    return ora_result_df

## gsea.py

In [12]:
# gsea.py
def generate_ranked_list(deg_data):
    """
    Generate a ranked gene list for GSEA.

    Parameters
    ----------
    deg_data : pandas.DataFrame
        Differential expression result table.
        Required columns:
            - Gene.symbol
            - logFC

    Returns
    -------
    pandas.DataFrame
        Gene list sorted by logFC (descending) without NaNs and duplicates.
    """

    # 检查输入表中是否包含必要列
    required_columns = ["Gene.symbol", "logFC"]
    for col in required_columns:
        if col not in deg_data.columns:
            raise ValueError(f"Missing required column: {col}")

    # 剔除没有基因名字（NaN）的非特异性行
    cleaned_data = deg_data.dropna(subset=["Gene.symbol"]).copy()

    # 3. 处理重复基因：按 logFC 的绝对值（变化幅度）从大到小排序，保留每个基因表达变化最显著的那一个探针
    cleaned_data["abs_logFC"] = cleaned_data["logFC"].abs()
    cleaned_data = cleaned_data.sort_values(by="abs_logFC", ascending=False)
    cleaned_data = cleaned_data.drop_duplicates(subset=["Gene.symbol"], keep="first")

    # 按 log2FoldChange 从大到小排
    # 上调最显著的基因位于顶部
    # 下调最显著的基因位于底部
    ranked_gene_list = (
        cleaned_data.sort_values(by="logFC", ascending=False)# 按照 log2FoldChange（Log2倍数变化）这一列进行降序排列（ascending=False）
        [["Gene.symbol", "logFC"]]# 只保留Gene_Symbol和log2FoldChange两列
        .reset_index(drop=True)# 按照log2FoldChange排序后原表格行的索引被打乱，drop=True丢弃原来的旧索引，重新从 0开始顺序编号。
    )

    return ranked_gene_list


def calculate_es(ranked_gene_list, pathway_genes):
    """
    Calculate simplified GSEA Enrichment Score (ES).

    Parameters
    ----------
    ranked_gene_list : pandas.DataFrame
        Output of generate_ranked_list()

    pathway_genes : set
        Genes belonging to a biological pathway

    Returns
    -------
    float
        Enrichment Score (ES)
    """

    # 保证 pathway_genes 为集合， 集合在底层基于哈希表实现。
    # Python查找集合时直接用哈希算法，不需要遍历，时间复杂度是O(1)
    pathway_genes = set(pathway_genes)

    # Ranked List总基因数
    total_genes = len(ranked_gene_list)

    # 实际出现在排序列表中的通路基因数
    pathway_hits = len(
        pathway_genes.intersection(
            ranked_gene_list["Gene.symbol"]
        )
    )

    # 如果没有任何通路基因出现在列表中，则无法进行富集分析
    if pathway_hits == 0:
        return 0.0

    # 极端情况保护
    if pathway_hits == total_genes:
        return 0.0

    # 命中通路基因时增加的步长
    hit_score = 1.0 / pathway_hits

    # 未命中通路基因时减少的步长
    miss_score = 1.0 / (total_genes - pathway_hits)

    # 当前运行富集分数
    current_es = 0.0

    # 记录最大偏离值
    max_es = 0.0

    # 从排序列表顶部开始遍历 (修复 Bug 1: 修改为 "Gene.symbol")
    for gene in ranked_gene_list["Gene.symbol"]:

        # 命中通路基因
        if gene in pathway_genes:
            current_es += hit_score
        # 非通路基因
        else:
            current_es -= miss_score

        # 记录绝对值最大的偏离位置（可能是正富集也可能是负富集）
        if abs(current_es) > abs(max_es):
            max_es = current_es

    return max_es


def run_gsea(deg_data, gene_sets):
    """
    Run simplified GSEA for all pathways.

    Parameters
    ----------
    deg_data : pandas.DataFrame
        Differential expression table

    gene_sets : dict
        Example:
        {
            "Apoptosis": {"TP53", "BAX", "CASP3"},
            "Cell_Cycle": {"CDK1", "CCNB1"}
        }

    Returns
    -------
    pandas.DataFrame
        Pathway enrichment results
    """
    # Step 1: 生成标准的基因排序列表
    ranked_gene_list = generate_ranked_list(deg_data)

    results = []

    # Step 2: 对每个通路计算ES
    for pathway_name, pathway_genes in gene_sets.items():
        es = calculate_es(ranked_gene_list, pathway_genes)

        results.append({
            "Pathway": pathway_name,
            "ES": es,
            "Gene_Count": len(pathway_genes)
        })

    # Step 3: 按ES绝对值降序排序，最显著富集的通路排在最前
    results_df = pd.DataFrame(results)
    results_df = (
        results_df
        .sort_values(by="ES", key=abs, ascending=False)
        .reset_index(drop=True)
    )

    return results_df

## plot.py

In [13]:
def plot_barplot(enrichment_results: pd.DataFrame, top_n: int = 10, output_path: str = "barplot.png"):
    """绘制 Top N 通路的富集条形图"""
    if enrichment_results.empty:
        print("警告：没有富集结果可用于绘制柱状图。")
        return
    
    # 1. 排序与筛选：按 FDR 从小到大排序，截取 Top N
    top_df = enrichment_results.sort_values(by="FDR").head(top_n).copy()
    
    # 2. 数据转换：计算 -log10(FDR)，让越显著的通路柱子越长
    top_df['-log10(FDR)'] = -np.log10(top_df['FDR'] + 1e-10) # 加上一个极小值防止 log(0)
    
    # 3. 绘图核心：初始化画布，绘制横向条形图
    # 3. 绘图核心：初始化画布，绘制横向条形图
    plt.figure(figsize=(10, 6))
    # 【修改点】：根据警告提示，增加 hue='pathway' 并关闭冗余的图例 legend=False
    sns.barplot(x='-log10(FDR)', y='pathway', data=top_df, palette='Reds_r', hue='pathway', legend=False)
    
    # 4. 图表修饰与输出
    plt.title("Top Enriched Pathways")
    plt.xlabel(r"$-\log_{10}(\text{FDR})$")
    plt.ylabel("Pathway")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()

def plot_dotplot(enrichment_results: pd.DataFrame, top_n: int = 10, output_path: str = "dotplot.png"):
    """绘制多维映射的气泡图"""
    if enrichment_results.empty:
        print("警告：没有富集结果可用于绘制气泡图。")
        return

    # 1. 排序与筛选
    top_df = enrichment_results.sort_values(by="FDR").head(top_n).copy()
    
    # 2. 绘图核心
    plt.figure(figsize=(10, 7))
    scatter = sns.scatterplot(
        x='GeneRatio',        # X轴：命中比例
        y='pathway',          # Y轴：通路名称
        size='Count',         # 气泡大小：命中基因数
        hue='FDR',            # 气泡颜色：FDR值
        data=top_df,
        palette='viridis',
        sizes=(50, 400)       # 控制气泡大小范围
    )
    
    # 3. 图表修饰与输出
    plt.title("Pathway Enrichment Dotplot")
    plt.xlabel("Gene Ratio")
    plt.ylabel("Pathway")
    
    # 移动图例到图表外部防止遮挡
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300)
    plt.close()

## 测试与绘图

In [14]:
# 1. 设定路径与读取数据
base_dir = Path.cwd()
gmt_file = base_dir / "data" / "h.all.v2026.1.Hs.symbols.gmt"
geo_file = base_dir / "data" / "GSE42568.top.table.tsv"

gene_sets = read_gmt(gmt_file)
deg_data = read_geo(geo_file)

# 2. 运行 ORA 分支 (演示)
ora_df = do_ora(deg_data, gene_sets)
if not ora_df.empty:
    ora_df['FDR'] = bh_fdr_correction(ora_df['P.Value'].values)

# 3. 绘图展示 (直接在 Notebook 中显示)
plot_barplot(ora_df, top_n=10, output_path="ora_barplot.png")
plot_dotplot(ora_df, top_n=10, output_path="ora_dotplot.png")

--- 背景基因库说明 ---
提取到全局背景基因 (Universe) 数量: 22189
提取到显著差异基因 (DEGs) 数量: 4687
原理解释: 仅使用在当前测序平台/微阵列中实际检测到的 22189 个基因作为超几何检验的背景，而不是整个人类基因组，能有效避免因技术偏倚导致的假阳性。
